## Setup

In [ ]:
# If you get import errors, install the packages from requirements.txt (see README.md)

import json, time, random
from pathlib import Path

import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageFile
import torch.nn.functional as F

# use the GPU if there is one (much faster), otherwise the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# fixed seed so that runs are reproducible and comparable
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

## Load data

In [ ]:
# Use the data folder "1_Case_AEC", which contains a subset of images from: 
# https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset 
# (Downloading and unzipping might take some time; we require many images to train the classifier)

# Change this path to where you unzipped the Kaggle dataset so that:
# DATA_DIR/
#   ├─ Cat/
#   └─ Dog/

DATA_DIR = "../data/1_Case_AEC/"  # relative to notebooks/ - every teammate puts the dataset there


# ImageFolder expects subfolders per class (Cat, Dog)
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate slightly broken JPEGs

# Return True only if `path` is a real, readable image.
#    What this does:
#    - Tries to open the file with Pillow (PIL) without fully decoding the pixels.
#    - `verify()` performs a quick integrity check (headers, structure).
#    - If the file is empty, corrupted, or not an image, Pillow raises an exception.
#    - We catch that and return False so bad files are filtered out safely.


def ok(path):
    # return True only if the file opens as an image (very short, very robust)
    try:
        Image.open(path).verify()
        return True
    except:
        return False

## Preprocessing

In [ ]:
# Use the same preprocessing that the pretrained weights expect (easy & reliable)
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

full_ds = ImageFolder(root=DATA_DIR, transform=preprocess, is_valid_file=ok)
print("Valid images:", len(full_ds), "| classes:", full_ds.classes)

## DataLoader; Train/Test split

In [ ]:
BATCH = 32  # how many images we process at once (fits in memory and speeds things up)
TRAIN_FRAC = 0.8  # 80% of the images for training, 20% for testing

# Split the dataset; the fixed generator gives the same split every time we run the notebook
n_train = int(TRAIN_FRAC * len(full_ds))
n_test = len(full_ds) - n_train
train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=torch.Generator().manual_seed(SEED))

# Training loader: shuffles the data each epoch so the model doesn't see batches in the same order
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

# Test loader: NO shuffling — we just want a stable, honest evaluation
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False)

# Quick sanity check: how many samples are in each split?
len(train_ds), len(test_ds)

## Build a small model (e.g., freeze ResNet-18, train only the last layer)

In [ ]:
# Pretrained backbone => We are building on an existing model with trained weights, this process is called transfer learning!
model = resnet18(weights=weights)

# Freeze all pretrained layers for speed/simplicity
for p in model.parameters():
    p.requires_grad = False

# Replace the final layer with a 2-class classifier (Cat/Dog)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Cross-entropy is the standard loss for classification (compares predicted class scores with the true label)
criterion = nn.CrossEntropyLoss()

# Select an optimizer below
# Then, we only pass model.fc.parameters() because we *froze* the pretrained backbone
# and want to train just the final layer. lr=1e-3 is a safe starting learning rate.
# Alternative: torch.optim.SGD(model.fc.parameters(), lr=1e-3, momentum=0.9) - compared in the experiments below
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

## Training

In [ ]:
# This might take several minutes due to complex/thorough training and the large number of images (>12.000)
model.train()  # put the model in *training mode* (enables layers like dropout/batchnorm to update)

# counters to track total loss and accuracy over the whole epoch
running_loss, total, correct = 0.0, 0, 0

for imgs, labels in train_loader:          # iterate over mini-batches of training data
    imgs, labels = imgs.to(device), labels.to(device)  # move data to GPU if available (else CPU)

    optimizer.zero_grad()                  # clear old gradients from the previous step
    logits = model(imgs)                   # forward pass: raw class scores for each image
    loss = criterion(logits, labels)       # compute how far predictions are from true labels

    loss.backward()                        # backpropagate: compute gradients
    optimizer.step()                       # update trainable parameters using the gradients

    # accumulate *sum* of losses so we can compute an average later
    running_loss += loss.item() * imgs.size(0)  # loss per sample × batch size

    # turn raw scores into predicted class indices (the highest score wins)
    preds = logits.argmax(1)

    # count how many predictions were correct in this batch
    correct += (preds == labels).sum().item()

    # count how many samples we've processed so far
    total += imgs.size(0)

# compute average loss and accuracy across the entire training set
train_loss = running_loss / total
train_acc  = correct / total

# TODO: Please compute other metrics and print them

print(f"Train — loss: {train_loss:.3f}, acc: {train_acc:.3f}")

## Evaluate on the test split

In [ ]:
model.eval()               # put the model in *evaluation mode* (disables dropout, uses running stats for batchnorm)
total, correct = 0, 0      # counters to compute accuracy across the whole test set

with torch.no_grad():      # turn off gradient tracking → faster and uses less memory (we're not training)
    for imgs, labels in test_loader:                 # iterate over mini-batches from the test split
        imgs, labels = imgs.to(device), labels.to(device)  # move data to GPU if available (else CPU)

        logits = model(imgs)        # forward pass only (no backward): get raw class scores
        preds = logits.argmax(1)    # pick the class index with the highest score for each image

        correct += (preds == labels).sum().item()  # count how many predictions were correct in this batch
        total   += imgs.size(0)                    # update how many test samples we've seen

# final test accuracy = (# correct predictions) / (total test samples)
test_acc = correct / total
print(f"Test — accuracy: {test_acc:.3f}")

# TODO: Please print metrics in a table and in graphs making the classification performance of the models visible  


## Experiments: train/test split sizes × optimizers

Each experiment trains a fresh model (frozen ResNet-18, new last layer) with the same seed, so the only
differences are the split size and the optimizer. Every run is saved to `results/<run_name>.json`
(format described in `PROJECT.md`) so that the metrics, table and graphs can be built from the files.

In [ ]:
def build_model():
    # same setup as above: pretrained backbone frozen, new 2-class last layer
    m = resnet18(weights=weights)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device)


def make_optimizer(name, params, lr):
    if name == "Adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")


def evaluate(model, loader):
    # returns average loss, accuracy and the per-image labels / predictions / P(Dog)
    model.eval()
    loss_sum, y_true, y_pred, y_prob = 0.0, [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss_sum += criterion(logits, labels).item() * imgs.size(0)
            y_true += labels.tolist()
            y_pred += logits.argmax(1).tolist()
            y_prob += F.softmax(logits, dim=1)[:, 1].tolist()
    acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
    return loss_sum / len(y_true), acc, y_true, y_pred, y_prob


def run_experiment(train_frac, optimizer_name, epochs=3, lr=1e-3):
    run_name = f"split{round(train_frac * 100)}_{optimizer_name.lower()}"
    print(f"\n=== {run_name} ===")
    set_seed()

    n_tr = int(train_frac * len(full_ds))
    tr_ds, te_ds = random_split(full_ds, [n_tr, len(full_ds) - n_tr], generator=torch.Generator().manual_seed(SEED))
    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH, shuffle=False)

    model = build_model()
    opt = make_optimizer(optimizer_name, model.fc.parameters(), lr)

    history, start = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, total, correct = 0.0, 0, 0
        for imgs, labels in tr_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            opt.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += imgs.size(0)

        test_loss, test_acc, y_true, y_pred, y_prob = evaluate(model, te_loader)
        history.append({"epoch": epoch, "train_loss": running_loss / total, "train_acc": correct / total,
                        "test_loss": test_loss, "test_acc": test_acc})
        print(f"epoch {epoch}: train loss {running_loss / total:.3f}, train acc {correct / total:.3f} | "
              f"test loss {test_loss:.3f}, test acc {test_acc:.3f}")

    result = {
        "run_name": run_name, "train_frac": train_frac, "optimizer": optimizer_name, "lr": lr,
        "epochs": epochs, "batch_size": BATCH, "seed": SEED, "n_train": len(tr_ds), "n_test": len(te_ds),
        "train_time_sec": round(time.time() - start, 1), "classes": full_ds.classes, "history": history,
        "test": {"y_true": y_true, "y_pred": y_pred, "y_prob": y_prob},
    }
    out_dir = Path("../results")
    out_dir.mkdir(exist_ok=True)
    with open(out_dir / f"{run_name}.json", "w") as f:
        json.dump(result, f)
    return result

In [ ]:
SPLITS = [0.8, 0.5]          # 80/20 and 50/50 train/test
OPTIMIZERS = ["Adam", "SGD"]
EPOCHS = 3

results = {}
for frac in SPLITS:
    for opt_name in OPTIMIZERS:
        r = run_experiment(frac, opt_name, epochs=EPOCHS)
        results[r["run_name"]] = r

## Metrics table and graphs

In [ ]:
# TODO (Issam): load results/*.json and show the metrics table + graphs using src/metrics_utils.py

## Test the model for single images

In [ ]:
class_names = getattr(full_ds, "classes", ["Cat", "Dog"])

def predict_image(img_path):
    """
    Load ONE image, apply the same preprocessing as training,
    run the model, and print Cat/Dog with confidence.
    """
    # 1) Open image and ensure 3-channel RGB
    img = Image.open(img_path).convert("RGB")

    # 2) Preprocess exactly like during training
    x = preprocess(img).unsqueeze(0).to(device)  # shape [1,3,H,W]

    # 3) Inference (no gradients)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = probs.max(dim=1)

    # 4) Human-readable output
    pred_label = class_names[int(pred_idx)]
    print(f"Prediction: {pred_label}  (confidence: {conf.item():.1%})")

In [ ]:
# Example use (end the path below with the folder and image you prefer):
predict_image(DATA_DIR + "Dog/995.jpg")